# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [ ]:
%pip install -q --upgrade pypdf gradio openai python-dotenv


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [1]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """Lee un PDF y devuelve todo su texto concatenado."""
    reader = PdfReader(pdf_path)
    paginas = []
    for page in reader.pages:
        paginas.append(page.extract_text() or "")
    return "\n".join(paginas)


document_text = extract_text_from_pdf("attention_is_all_you_need.pdf")
print(f"Paginas: {len(PdfReader('attention_is_all_you_need.pdf').pages)}")
print(f"Caracteres extraidos: {len(document_text)}")
print(document_text[:500])


Paginas: 15
Caracteres extraidos: 39510
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
...

> **Nota de ejecucion:** el Paso 1 (extraccion del PDF) se ejecuto aqui de verdad: el paper tiene 15 paginas y `extract_text_from_pdf` recupera 39,510 caracteres de texto. Los pasos 2 a 4 (cliente de NVIDIA NIM y la app de Gradio) necesitan una `NVIDIA_API_KEY` personal (gratuita en https://build.nvidia.com/settings/api-keys) que no esta disponible en este entorno, asi que **no se ejecutaron** aqui; el codigo esta completo y sigue exactamente las instrucciones de cada paso. Para probarlo: crea un `.env` con `NVIDIA_API_KEY="tu_key"` en esta carpeta y corre `Restart & Run All`.

## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

assert os.getenv("NVIDIA_API_KEY"), (
    "Falta NVIDIA_API_KEY. Crea un archivo .env con NVIDIA_API_KEY=\"tu_key\" "
    "(ver https://build.nvidia.com/settings/api-keys)."
)
print("NVIDIA_API_KEY cargada:", bool(os.getenv("NVIDIA_API_KEY")))

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [ ]:
def build_system_prompt(document_text: str) -> str:
    """Instruye al modelo a responder solo con base en el documento entregado."""
    return (
        "Eres un asistente experto en el siguiente paper academico. "
        "Responde UNICAMENTE usando la informacion contenida entre los delimitadores "
        "<documento> y </documento>. No uses conocimiento externo. "
        "Si la respuesta no esta en el documento, responde exactamente: "
        "\"No encuentro esa informacion en el documento\".\n\n"
        f"<documento>\n{document_text}\n</documento>"
    )


def history_to_messages(history):
    """Convierte el historial de Gradio (lista de dicts role/content) a mensajes de chat."""
    messages = []
    for turn in history:
        role = turn.get("role")
        content = turn.get("content")
        if role in ("user", "assistant") and content:
            messages.append({"role": role, "content": content})
    return messages


def chat_con_documento(message, history, document_text):
    """Responde en streaming usando el documento como unico contexto."""
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    respuesta_completa = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta_completa += texto
            yield respuesta_completa


## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [ ]:
import gradio as gr

document_text = extract_text_from_pdf("attention_is_all_you_need.pdf")
print(f"Documento cargado: {len(document_text)} caracteres")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    title="Asistente del paper: Attention Is All You Need",
    description=(
        "Pregunta sobre 'Attention Is All You Need' (Vaswani et al., 2017). "
        "Las respuestas se basan unicamente en el texto del documento."
    ),
    examples=[
        "¿Cual es la arquitectura principal propuesta en el paper?",
        "¿Que es el mecanismo de atencion?",
        "¿Cuantas capas tiene el encoder del modelo base?",
    ],
    type="messages",
)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)